# IADD YOLOv11 - run8: AdamW + cosine LR

## 1. GPU / install / mount

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
MODEL    = 'yolo11s.pt'
IMGSZ    = 1280
BATCH    = 12
EPOCHS   = 150
PATIENCE = 10
RUN_NAME = 'run8'

# all training settings (recorded in args.yaml)
OPTIM    = 'AdamW'
LR0      = 0.001
COS_LR   = True

# augmentations: better generalization; less overfitting
MIXUP    = 0.1
HSV_V    = 0.4
DEGREES  = 5.0

DRIVE    = '/content/drive/MyDrive'
RUNS     = f'{DRIVE}/iadd_runs'
DS       = '/content/iadd_subset_v5'
DATA     = f'{DS}/data.yaml'

## 3. Unzip the dataset

In [ ]:
!rm -rf /content/iadd_subset_v5
!unzip -q "/content/drive/MyDrive/iadd_subset_v5.zip" -d /content

## 4. Train

In [ ]:
from ultralytics import YOLO
import os

CKPT = f'{RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT)
    model.train(resume=True)
else:
    print('fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH, '| opt', OPTIM)
    model = YOLO(MODEL)
    model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=RUN_NAME, exist_ok=True, plots=True,
        optimizer=OPTIM, lr0=LR0, cos_lr=COS_LR, mixup=MIXUP, hsv_v=HSV_V, degrees=DEGREES,
    )

## 5. Convergence check

In [ ]:
import pandas as pd

df = pd.read_csv(f'{RUNS}/{RUN_NAME}/results.csv')
df.columns = [c.strip() for c in df.columns]
fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']
best_i = int(fit.idxmax())
since = len(fit) - 1 - best_i
print(f'epochs done: {len(df)}   best at epoch {int(df["epoch"].iloc[best_i])}   ({since} ago)')
print('\n>>> CONVERGED - safe to STOP <<<' if since >= PATIENCE else '>>> still improving <<<')

## 6. Finisher: fine-tune 10 low-LR epochs

In [ ]:
from ultralytics import YOLO
import os

FT_CKPT = f'{RUNS}/{RUN_NAME}_ft/weights/last.pt'
if os.path.exists(FT_CKPT):
    print('resuming finish ->', FT_CKPT)
    YOLO(FT_CKPT).train(resume=True)
else:
    YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt').train(
        data=DATA, epochs=10, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=f'{RUN_NAME}_ft', exist_ok=True, plots=True,
        optimizer='َAdamW', lr0=0.0001, lrf=0.1, cos_lr=COS_LR, warmup_epochs=0.0,
        mosaic=0.0, mixup=0.0, hsv_v=HSV_V, degrees=DEGREES)

## 7. Evaluation: main vs FT, mAP, confusion matrix

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil

TAG = f'{RUN_NAME}_eval'
cands = [('main', f'{RUNS}/{RUN_NAME}/weights/best.pt')]
ft = f'{RUNS}/{RUN_NAME}_ft/weights/best.pt'
if os.path.exists(ft): cands.append(('ft', ft))

res = {}
for name, w in cands:
    m = YOLO(w)ّ
    v = m.val(data=DATA, split='val',  plots=True, project=RUNS, name=f'{TAG}_{name}_val',  exist_ok=True)
    t = m.val(data=DATA, split='test', plots=True, project=RUNS, name=f'{TAG}_{name}_test', exist_ok=True)
    res[name] = (m, v, t)
    print('%-4s VAL mAP50=%.4f mAP50-95=%.4f | TEST mAP50=%.4f mAP50-95=%.4f'
          % (name, v.box.map50, v.box.map, t.box.map50, t.box.map))

win = max(res, key=lambda k: res[k][1].box.map50)
best, mv, mt = res[win]
print('>>> official model for %s: %s (val mAP50=%.4f, test mAP50=%.4f)'
      % (RUN_NAME, win, mv.box.map50, mt.box.map50))

CONF = 0.25
for x, y, _, yl in mv.curves_results:
    if yl == 'F1':
        px = np.asarray(x); y = np.asarray(y); f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating confidence (val F1) =', CONF)
for split in ('val', 'test'):
    best.val(data=DATA, split=split, conf=CONF, plots=True, project=RUNS, name=f'{TAG}_{split}_tmp', exist_ok=True)
    for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
        shutil.copy(f'{RUNS}/{TAG}_{split}_tmp/{f}', f'{RUNS}/{TAG}_{win}_{split}/{f}')
    shutil.rmtree(f'{RUNS}/{TAG}_{split}_tmp')
print('done. winner=%s  matrix at conf %.3f' % (win, CONF))